# OneRef sur Colab
Profil **8B** : GPU **L4** obligatoire. Profil 4B : T4 ou L4. Lance les cellules une par une ; les sorties sont sur Drive donc un run coupé **reprend** là où il s'est arrêté.

In [ ]:
# 1. GPU + profil de modeles + arguments vLLM
PROFILE = "8B"   # "8B" : Qwen3-VL-8B, InternVL3.5-8B, Gemma 4 E4B (L4) ; "4B" : modeles 4B (T4 possible)
import subprocess, json, os
gpu = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],capture_output=True,text=True).stdout.strip()
print(gpu)
T4 = "T4" in gpu
if PROFILE == "8B" and (T4 or not gpu):
    raise SystemExit("Profil 8B : poids d'environ 16 Go, il faut un GPU L4 (ou mieux). Change le type d'execution.")
# 8B sur L4 (24 Go) : contexte reduit (entrees observees <= 10,5k tokens) pour laisser le cache KV
max_len, mem = ("12288", "0.92") if PROFILE == "8B" else ("16384", "0.85")
vllm_args = ["--max-model-len",max_len,"--gpu-memory-utilization",mem,"--trust-remote-code","--enable-auto-tool-choice","--tool-call-parser","hermes"]
if T4:
    vllm_args += ["--dtype","half","--enforce-eager"]   # pas de bf16 sur T4 ; eager = moins de memoire
os.environ["ONEREF_VLLM_ARGS"] = json.dumps(vllm_args)

In [ ]:
# 2. Drive (donnees/sorties) + code depuis GitHub (branche Test_implementation)
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/OneRef"
os.makedirs(DRIVE, exist_ok=True)
!rm -rf /content/OneRef && git clone -q --depth 1 -b Test_implementation https://github.com/RVilchezzz/OneRef.git /content/OneRef
!rm -rf /content/OneRef_Colab && cp -r /content/OneRef/Colab /content/OneRef_Colab && cp -r /content/OneRef/configs /content/OneRef_Colab/configs
%cd /content/OneRef_Colab


In [ ]:
# 3. Installation (vLLM peut remplacer torch : si Colab le demande, redémarre le runtime puis relance depuis la cellule 1)
!pip install -q -r requirements_colab.txt

In [ ]:
# 4. Hugging Face (secret Colab "HF_TOKEN" ; accepter la licence DINOv3 sur le Hub)
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [ ]:
# 5. Variables d'environnement du projet
os.environ.update({
    "ONEREF_MMAD_ROOT": f"{DRIVE}/MMAD",
    "ONEREF_MMAD_JSON": f"{DRIVE}/MMAD/mmad.json",
    "ONEREF_OUTPUTS_DIR": f"{DRIVE}/outputs",
    "ONEREF_SPLITS_DIR": f"{DRIVE}/splits",
    "ONEREF_LLM_PROVIDER": "vllm",          # le code par défaut utilise un API hébergée ; ici : vLLM local, un serveur par passe
    "ONEREF_LLM_BASE_URL": "http://localhost:8000/v1",
    "ONEREF_LLM_API_KEY": "EMPTY",
    "ONEREF_VLLM_AUTOSTART": "true",
    "ONEREF_DEVICE": "cuda",
    "PYTHONPATH": "/content/OneRef_Colab",
})
# MMAD : "subset" (N images du split dev + leurs references, ~quelques Mo : recommande pour tester) | "sample" (zip DS-MVTec ; ses
# references sont dans MVTec-AD.zip donc --available-only ne trouve rien) | "full" (~28 Go) | "none" (deja sur Drive)
MMAD_MODE = "subset"
N_SAMPLE = 50
SAMPLE_ZIPS = ["DS-MVTec.zip"]   # DS-MVTec 1.6 Go, VisA 1.9 Go, MVTec-AD 5.3 Go, MVTec-LOCO 6.1 Go, GoodsAD 13.3 Go
OPT = "--available-only" if MMAD_MODE == "sample" else (f"--sample {N_SAMPLE}" if MMAD_MODE == "subset" else "")   # n'utiliser que les images téléchargées

# Config du profil (clonée avec le code) : modeles, judge, calibration de l'expert
import shlex
CONFIG = {"8B": "configs/2026-11-19_8B_colab_template.env", "4B": "configs/2026-11-19_4B_colab_vllm.env"}[PROFILE]
for line in open(CONFIG, encoding="utf-8"):
    line = line.strip()
    if line and not line.startswith("#"):
        k, v = line.split("=", 1); os.environ[k] = shlex.split(v)[0] if v else ""

# Profil 4B : Gemma 4 E2B ne demarre pas sur T4 (shared memory Triton) : critic et judge remplaces
os.environ.pop("ONEREF_AGGREGATOR_MODEL", None)   # aggregator = vote pondere
if T4 and PROFILE == "4B":
    os.environ.update({
        "ONEREF_CRITIC_MODEL": "Qwen/Qwen3-VL-4B-Instruct",
        "ONEREF_JUDGE_MODEL": "mistralai/Ministral-3-3B-Instruct-2512-BF16",
        "ONEREF_CRITIC_EXTRA_BODY": "{}",
        "ONEREF_JUDGE_EXTRA_BODY": "{}",
    })
print({k: v for k, v in os.environ.items() if k.startswith("ONEREF_") and k.endswith("_MODEL")})

In [ ]:
# 5b. Telecharger MMAD sur Drive (UNE seule fois). "subset" : mmad.json + N images dev et leurs references ; "sample" : un zip ; "full" : ~28 Go.
# Sauter cette cellule si MMAD est deja dans MyDrive/OneRef/MMAD (MMAD_MODE = "none").
if MMAD_MODE == "subset":
    from download_mmad import download_mmad
    download_mmad(f"{DRIVE}/MMAD", [])   # mmad.json seul
    os.makedirs(f"{DRIVE}/splits", exist_ok=True)
    !python /content/OneRef/scripts/fetch_mmad_subset.py --split dev --sample {N_SAMPLE} --root {DRIVE}/MMAD --splits-dir {DRIVE}/splits
elif MMAD_MODE in ("sample", "full"):
    from download_mmad import ALL_ZIPS, download_mmad
    download_mmad(f"{DRIVE}/MMAD", ALL_ZIPS if MMAD_MODE == "full" else SAMPLE_ZIPS)
else:
    print("MMAD_MODE = none : rien a telecharger")


In [ ]:
# 6. Smoke test sans GPU
!python -m pytest -q
!python -m inspector.main --split dev --mock --limit 20 --run-name colab_smoke {OPT}


In [ ]:
# 6b. Demarrer chaque modele une fois avant la sonde (echec ici = rien de perdu)
if not T4:
    from openai import OpenAI
    from inspector.agents.llm import VLLMServer
    from inspector.config.settings import Settings
    s = Settings()
    for name in dict.fromkeys([s.reader_a_model, s.reader_b_model, s.critic_model, s.judge_model]):
        extra = s.critic_extra_body if name == s.critic_model else None
        srv = VLLMServer(s, name)
        try:
            srv.start()
            r = OpenAI(base_url=s.llm_base_url, api_key="EMPTY").chat.completions.create(
                model=name, max_tokens=256, extra_body=extra,
                messages=[{"role": "user", "content": 'Reply only with this JSON: {"ok": true}'}])
            print(name, "OK :", r.choices[0].message.content)
        finally:
            srv.stop()

In [ ]:
# 7. Sonde : N_SAMPLE images dev, vrais modeles, puis controle des seuils (VERDICT)
RUN = f"probe_{PROFILE}"
!python -m inspector.main --split dev --run-name {RUN} {OPT}
!python /content/OneRef/scripts/check_run.py {DRIVE}/outputs/{RUN}

In [ ]:
# 8. Pipeline complete sur le dev (835 images). Reprenable : relance la cellule apres une coupure.
# Ne demarre que si la sonde passe les seuils. Verifie aussi les unites de calcul restantes (panneau Ressources).
import shutil, subprocess, sys
probe, full = f"{DRIVE}/outputs/probe_{PROFILE}", f"{DRIVE}/outputs/dev_full_S_{PROFILE}"
if subprocess.run([sys.executable, "/content/OneRef/scripts/check_run.py", probe]).returncode:
    raise SystemExit("La sonde ne passe pas les seuils : dev complet non lance.")
!python /content/OneRef/scripts/fetch_mmad_subset.py --split dev --root {DRIVE}/MMAD --splits-dir /content/OneRef/data/splits
if not os.path.exists(full):
    shutil.copytree(probe, full)   # memes modeles et reglages, images du dev : leur cache est reutilise
!python -m inspector.main --split dev --run-name dev_full_S_{PROFILE}

In [ ]:
# 9. Bilan du dev complet
!python /content/OneRef/scripts/check_run.py {DRIVE}/outputs/dev_full_S_{PROFILE}

## Test (une seule fois, config figée)
Ne lance **jamais** le split test pour régler des paramètres.

In [ ]:
# !python -m inspector.main --split test --frozen-config --run-name test01 {OPT}